# Vaardigheden week 4: grafieken die de waarheid vertellen

Dit notebook hoort bij het vaardighedenspoor. Het gebruikt verzonnen data over een busbedrijf, een bakkerij en een hardloper, zodat de vaardigheden los blijven van het project van deze week.

Je doet de vijfsecondentest, maakt staaf- en lijngrafieken met de eerlijkheidsset, meet hoeveel een afgesneden as overdrijft, betrapt een AI-grafiek die misleidt, en bouwt met opzet een misleidende grafiek, zodat je er een herkent.

**Zo werk je ermee:** voer elke cel op volgorde uit. Als een cel je vraagt iets te schrijven, dubbelklik je op de tekstcel en typ je.

## 1. Drie verzonnen tabellen

De eerste cel maakt drie kleine tabellen vanuit een seed, zodat iedereen dezelfde getallen krijgt.

- De bustabel bevat het aandeel bussen dat op tijd aankwam, elke maand gedurende drie jaar.
- De bakkerijtabel bevat het gemiddelde aantal verkochte broden per dag in vijf filialen.
- De hardlooptabel bevat de tijd van één hardloper voor 5 km, één keer per week.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(404)

# Een verzonnen busbedrijf: het aandeel bussen dat op tijd aankwam, elke maand gedurende drie jaar
t = np.arange(36)
trend = np.where(t < 9, 83.0 + 0.14 * t, 84.26 - 0.09 * (t - 9))
bus = pd.DataFrame({
    "month": pd.date_range("2023-01-01", periods=36, freq="MS"),
    "on_time_pct": (trend + rng.normal(0, 0.3, 36)).round(1),
})

# Een verzonnen bakkerij: verkochte broden per dag in vijf filialen, gemiddeld over 30 dagen
loaves = pd.DataFrame(rng.poisson([71, 72, 74, 73, 75], size=(30, 5)), columns=list("ABCDE"))
bakery = loaves.mean().round(1).rename("loaves_per_day")

# Een verzonnen hardloper: de tijd voor 5 km, één keer per week gedurende 16 weken
runner = pd.DataFrame({
    "week": np.arange(1, 17),
    "minutes": (28.4 - 0.13 * np.arange(16) + rng.normal(0, 0.15, 16)).round(2),
})

print(bus.head())
print()
print(bakery)

## 2. De vijfsecondentest

Een grafiek werkt op je in voordat je nadenkt. Voer de volgende cel uit, kijk vijf seconden naar de grafiek en kijk dan weg. Lees de getallen op de as niet.

Schrijf op wat je nu gelooft over de bussen voordat je verder leest.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bus["month"], bus["on_time_pct"], marker="o", markersize=3)
ax.set_title("Bussen op tijd, per maand")
ax.set_xlabel("maand")
ax.set_ylabel("bussen op tijd (%)")
default_ax = ax                      # bewaar de assen, zodat we ze later kunnen meten
plt.tight_layout()
plt.show()

**Mijn eerste indruk (dubbelklik om te schrijven):**

**Het kenmerk van de grafiek dat me daar bracht:**

## 3. Dezelfde getallen op een as vanaf nul

De volgende grafiek tekent precies dezelfde getallen. Het enige verschil is dat de y-as loopt van 0 tot 100 procent. Vergelijk hem met je eerste indruk.

Als code de y-as nooit instelt, laat matplotlib de as net onder de kleinste waarde beginnen. Door die standaard vult elke verandering de hele hoogte van de grafiek. De laatste regel van de cel drukt af waar de as van de eerste grafiek begon.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(bus["month"], bus["on_time_pct"], marker="o", markersize=3)
ax.set_ylim(0, 100)
ax.set_title("De meeste bussen zijn op tijd, en het aandeel is iets gedaald")
ax.set_xlabel("maand")
ax.set_ylabel("bussen op tijd (%)")
plt.tight_layout()
plt.show()

print("De as van de eerste grafiek begint bij:", round(default_ax.get_ylim()[0], 2))
print("De as van deze grafiek begint bij:", ax.get_ylim()[0])

## 4. Twee manieren om te zeggen hoeveel

Een verandering tussen twee percentages kun je op twee manieren uitdrukken.

- **Procentpunten** zijn het gewone verschil tussen de twee percentages.
- **Procent** is de verandering als deel van de beginwaarde.

De cel hieronder werkt eerst een klein voorbeeld uit en meet daarna het aandeel bussen op tijd van de eerste tot de laatste maand. Zeg altijd welke van de twee je bedoelt.

In [ ]:
before, after = 50, 40
print(f"Van {before}% naar {after}%: {after - before} punten, dat is {(after - before) / before * 100:.1f} procent")
print()

first = bus["on_time_pct"].iloc[0]
last = bus["on_time_pct"].iloc[-1]
points = last - first                      # procentpunten: het gewone verschil
percent = (last - first) / first * 100     # procent: de verandering als deel van de beginwaarde
print("Eerste maand:", first, "% op tijd. Laatste maand:", last, "% op tijd.")
print("Verandering in procentpunten:", round(points, 1))
print("Verandering in procent:", round(percent, 2))
print("Verandering per jaar, in punten:", round(points / (len(bus) - 1) * 12, 2))

### Maak een grafiek van de verandering zelf

Een lijn van het niveau beantwoordt de vraag *hoeveel is er?* Als de vraag is *hoe snel verandert het?*, maak je een grafiek van de verandering zelf.

De methode `.diff()` trekt van elke waarde de waarde ervoor af. Hieronder middelt het notebook elk jaar, neemt het de verandering van het ene jaar op het volgende, en tekent het die veranderingen als staven vanaf nul. Een staaf onder nul is een daling.

In [ ]:
yearly = bus.groupby(bus["month"].dt.year)["on_time_pct"].mean().round(2)
change = yearly.diff()
print(pd.DataFrame({"jaargemiddelde": yearly, "verandering t.o.v. het jaar ervoor": change}))

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(change.index[1:].astype(str), change.iloc[1:])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Het aandeel op tijd daalde in elk van de laatste twee jaar")
ax.set_xlabel("jaar")
ax.set_ylabel("verandering t.o.v. het jaar ervoor (punten)")
plt.tight_layout()
plt.show()

## 5. Staven vergelijken hoeveelheden

Een staaf laat zijn hoeveelheid zien met zijn lengte, dus de waarde-as van een staafgrafiek begint bij nul. matplotlib laat staven bij nul beginnen, tenzij de code iets anders zegt.

De grafiek hieronder heeft de volledige eerlijkheidsset:

- een titel die de bewering uitspreekt, niet alleen het onderwerp;
- aslabels met hun eenheden;
- een notitie die de bron noemt;
- staven die bij nul beginnen.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(bakery.index, bakery.values)
ax.set_title("De vijf filialen verkopen bijna evenveel brood")
ax.set_xlabel("filiaal")
ax.set_ylabel("verkochte broden per dag (gemiddelde over 30 dagen)")
ax.text(1.0, -0.3, "Bron: verzonnen data, seed 404", transform=ax.transAxes, ha="right", fontsize=8)
plt.tight_layout()
plt.show()
print("De as van deze grafiek begint bij:", ax.get_ylim()[0])

## 6. Meet de overdrijving

De overdrijvingsfactor zegt hoeveel keer groter een grafiek een verandering laat lijken dan ze werkelijk is.

- De **echte verandering** meet je vanaf nul: de verandering gedeeld door de beginwaarde.
- De **getekende verandering** meet je vanaf de onderkant van de as, want dat ziet het oog: de verandering gedeeld door de hoogte van de beginwaarde boven de onderkant van de as.
- De factor is de getekende verandering gedeeld door de echte verandering. Een grafiek met een as vanaf nul heeft factor 1.

In [ ]:
def exaggeration(base, other, bottom):
    """Hoeveel keer groter een grafiek de verandering van base naar other laat lijken dan ze is.
    bottom is de waarde waar de as van de grafiek begint."""
    real = (other - base) / base               # de verandering als deel van de beginwaarde
    drawn = (other - base) / (base - bottom)   # de verandering als deel van de getekende hoogte boven de as
    return drawn / real

print("Een daling van 100 naar 95 op een as die bij 90 begint: factor", round(exaggeration(100, 95, 90), 1))
print("Echte verandering:", (95 - 100) / 100 * 100, "procent. Getekende verandering:", (95 - 100) / (100 - 90) * 100, "procent van de hoogte boven de as.")
print("Een daling van 50 naar 40 op een as die bij 30 begint: factor", round(exaggeration(50, 40, 30), 1))
print("Echte verandering:", (40 - 50) / 50 * 100, "procent. Getekende verandering:", (40 - 50) / (50 - 30) * 100, "procent van de hoogte boven de as.")
print()

bottom = default_ax.get_ylim()[0]
print("Echte verandering, eerste tot laatste maand:", round(percent, 2), "procent")
print("Getekende verandering in de eerste grafiek:", round((last - first) / (first - bottom) * 100, 1), "procent van de beginhoogte")
print("Overdrijvingsfactor van de eerste grafiek:", round(exaggeration(first, last, bottom), 1))
print("Overdrijvingsfactor van de grafiek vanaf nul:", round(exaggeration(first, last, 0), 1))

## 7. Wanneer de AI zich vergist

Stel dat je een AI-assistent vraagt: *"Maak een staafgrafiek van de broden per dag per filiaal waarin de verschillen goed te zien zijn, en beschrijf wat hij laat zien."* De assistent geeft de cel hieronder terug, met deze beschrijving: *"Filiaal E verkoopt ongeveer vier keer zoveel brood als filiaal B."*

Voer de cel uit en beslis dan of de beschrijving klopt.

In [ ]:
# Geschreven door een AI-assistent
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(bakery.index, bakery.values, color="tab:orange")
ax.set_ylim(bakery.min() - 1, bakery.max() + 1)   # inzoomen zodat de verschillen goed te zien zijn
ax.set_title("Filiaal E verkoopt veel meer dan filiaal B")
ax.set_xlabel("filiaal")
ax.set_ylabel("broden per dag")
ai_ax = ax
plt.tight_layout()
plt.show()

De grafiek ziet er dramatisch uit, omdat de code de as één brood onder de kortste staaf laat beginnen. Daarna beschreef de assistent de afbeelding die hij tekende, niet de bakkerij. De AI deed wat er gevraagd werd, en juist de vraag was de valkuil: *maak de verschillen goed zichtbaar* nodigt uit tot een afgesneden as.

De volgende cel meet de bakkerij en de afbeelding naast elkaar.

In [ ]:
hi, lo = bakery.idxmax(), bakery.idxmin()
bottom = ai_ax.get_ylim()[0]
real_ratio = bakery[hi] / bakery[lo]
drawn_ratio = (bakery[hi] - bottom) / (bakery[lo] - bottom)
print("Hoogste staaf:", hi, bakery[hi], "broden. Kortste staaf:", lo, bakery[lo], "broden.")
print("Echte verhouding tussen de twee filialen:", round(real_ratio, 2))
print("Verhouding van de staafhoogtes zoals getekend:", round(drawn_ratio, 1))
print("Overdrijvingsfactor van de grafiek van de AI:", round(exaggeration(bakery[lo], bakery[hi], bottom), 1))
print("De as van de AI loopt van", round(ai_ax.get_ylim()[0], 1), "tot", round(ai_ax.get_ylim()[1], 1), "broden")

# CHECK: de "vier keer" van de AI is wat de grafiek tekent, niet wat de bakkerij verkoopt
assert (hi, lo) == ("E", "B")
assert round(drawn_ratio) == 4
assert real_ratio < 1.1
print("CHECK geslaagd")

## 8. Wanneer een ingezoomde as eerlijk is

Een lijngrafiek mag zijn as boven nul laten beginnen als nul niets betekent voor de vraag, of als een kleine verandering veel uitmaakt. De grafiek moet dat dan wel zeggen.

Niemand loopt 5 km in nul minuten, en een minuut is voor een hardloper veel winst. De grafiek hieronder zoomt in en markeert dat twee keer: de titel noemt de echte verandering in minuten en waar de as begint, en een notitie op de grafiek zegt dat de as niet bij nul begint.

Staven krijgen deze uitzondering nooit, want hun lengte is de boodschap.

In [ ]:
start, end = runner["minutes"].iloc[0], runner["minutes"].iloc[-1]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(runner["week"], runner["minutes"], marker="o")
ax.set_ylim(25, 29)
ax.set_title(f"Tijd op 5 km daalde {start - end:.1f} minuten in 16 weken (as begint bij 25 minuten)")
ax.set_xlabel("week")
ax.set_ylabel("minuten voor 5 km")
ax.text(0.99, 0.92, "as begint niet bij nul", transform=ax.transAxes, ha="right", color="firebrick")
plt.tight_layout()
plt.show()
print("Eerste week:", start, "minuten. Laatste week:", end, "minuten.")
print("Verandering:", round(end - start, 2), "minuten, dat is", round((end - start) / start * 100, 1), "procent")
print("De as van deze grafiek begint bij:", ax.get_ylim()[0], "minuten")

## 9. Jouw beurt 1: meet vanaf de beste maand

Waar je begint met meten, verandert het antwoord. Het aandeel bussen op tijd steeg een aantal maanden voordat het daalde.

Zoek de maand met het hoogste aandeel op tijd en sla die op in `best_month`. Sla daarna de verandering van die maand tot de laatste maand op in `points_from_best` (in procentpunten) en `percent_from_best` (in procent). Gebruik je AI-assistent als je wilt, en lees de code regel voor regel voordat je hem uitvoert.

De CHECK-cel drukt ook de verandering vanaf de eerste maand af, zodat je de twee periodes kunt vergelijken.

In [ ]:
# jouw beurt 1: zoek de beste maand en meet de verandering van die maand tot de laatste maand
best_month = None          # de maand met het hoogste aandeel op tijd
points_from_best = None    # verandering in procentpunten, beste maand tot laatste maand
percent_from_best = None   # verandering in procent, beste maand tot laatste maand

In [ ]:
# CHECK 1
assert best_month is not None, "best_month is nog leeg"
top = bus.loc[bus["on_time_pct"].idxmax()]
assert pd.Timestamp(best_month) == top["month"], "best_month is niet de maand met het hoogste aandeel"
assert abs(points_from_best - (last - top["on_time_pct"])) < 0.01
assert abs(percent_from_best - (last - top["on_time_pct"]) / top["on_time_pct"] * 100) < 0.01
print("CHECK 1 geslaagd:", top["month"].strftime("%Y-%m"), round(points_from_best, 1), "punten,", round(percent_from_best, 1), "procent")
print("Vanaf de eerste maand was de verandering", round(points, 1), "punten.")

## 10. Jouw beurt 2: bouw met opzet de misleidende grafiek

Zelf de leugen maken is de snelste manier om te leren hem te herkennen.

Teken de busreeks als lijngrafiek, met de y-as zo afgesneden dat de daling van de beste maand tot de laatste maand minstens 20 keer groter lijkt dan ze is. Houd elk punt op de grafiek. Geef hem een dramatische titel, zet er met `ax.text` de woorden *misleidend, met opzet* op als stempel, en sla de assen op in `cut_ax`.

Een misleidende grafiek verlaat het notebook nooit zonder zijn stempel.

In [ ]:
# jouw beurt 2: een lijngrafiek van het aandeel op tijd met een afgesneden as en een stempel, opgeslagen in cut_ax
cut_ax = None

In [ ]:
# CHECK 2
assert cut_ax is not None, "cut_ax is nog leeg"
stamp = " ".join(t.get_text().lower() for t in cut_ax.texts)
assert "misleading" in stamp or "misleidend" in stamp, "zet de woorden misleidend, met opzet als stempel op de grafiek"
assert cut_ax.get_ylim()[0] < bus["on_time_pct"].min(), "houd elk punt op de grafiek"
cut_factor = exaggeration(bus["on_time_pct"].max(), last, cut_ax.get_ylim()[0])
assert cut_factor >= 20, f"de factor is maar {cut_factor:.1f}; snijd de as dichter bij de data af"
print("CHECK 2 geslaagd: de as begint bij", round(cut_ax.get_ylim()[0], 2), "en de factor is", round(cut_factor, 1))

## 11. Jouw beurt 3: verbeter de grafiek van de AI

Teken de bakkerijgrafiek uit deel 7 opnieuw, maar nu eerlijk. Houd dezelfde getallen en gebruik staande staven (`ax.bar`). Laat de staven bij nul beginnen, schrijf een titel die zegt wat de grafiek echt laat zien, benoem beide assen, en sla de assen op in `fixed_ax`.

Schrijf daarna in de tekstcel hieronder één zin die een bedrijfsleider van de bakkerij kan vertrouwen. Gebruik de echte verhouding uit deel 7.

In [ ]:
# jouw beurt 3: een eerlijke staafgrafiek van de bakkerij, opgeslagen in fixed_ax
fixed_ax = None

In [ ]:
# CHECK 3
assert fixed_ax is not None, "fixed_ax is nog leeg"
assert fixed_ax.get_ylim()[0] == 0, "de staven moeten bij nul beginnen"
assert fixed_ax.get_title().strip() and fixed_ax.get_title() != ai_ax.get_title(), "schrijf een nieuwe titel die zegt wat de grafiek laat zien"
assert fixed_ax.get_ylabel().strip(), "benoem de y-as met zijn eenheid"
heights = sorted(round(float(p.get_height()), 1) for p in fixed_ax.patches)
assert heights == sorted(bakery.round(1).tolist()), "houd dezelfde getallen als in de grafiek van de AI"
print("CHECK 3 geslaagd: de overdrijvingsfactor is nu", round(exaggeration(bakery.min(), bakery.max(), fixed_ax.get_ylim()[0]), 1))

**Mijn zin voor de bedrijfsleider van de bakkerij (dubbelklik om te schrijven):**

## 12. Voordat je het notebook sluit

Doe de vijfsecondentest uit deel 2 met twee mensen, weg van het scherm. Laat elke persoon de eerste busgrafiek vijf seconden zien, schrijf op wat die persoon geloofde, en benoem het kenmerk dat hem of haar daar bracht. Laat daarna de grafiek uit deel 3 zien en schrijf op wat er veranderde.

## 13. Verdieping: de jacht op de periode

Deze opdracht is moeilijker dan de drie ervoor. In deel 9 mat je vanaf twee beginpunten. Hier meet je vanaf elk beginpunt, om te zien hoeveel ruimte een grafiekmaker heeft om het verhaal te kiezen.

1. Sla in `window_changes` de verandering in procentpunten op van elke maand tot de laatste maand, voor elke maand behalve de laatste. Gebruik de maand als index.
2. Sla in `harshest_start` de beginmaand op die de grootste daling geeft, en in `kindest_start` de beginmaand die de kleinste daling of de grootste stijging geeft.
3. Sla in `n_rises` het aantal beginmaanden op waarvandaan het aandeel lijkt te stijgen.
4. Teken `window_changes` als staven vanaf nul, met een titel met een bewering en benoemde assen, en sla de assen op in `window_ax`.

Schrijf daarna in de tekstcel hieronder twee koppen, één vanaf `harshest_start` en één vanaf `kindest_start`. Beide moeten kloppen. Schrijf een derde zin die een zorgvuldige lezer kan vertrouwen.

In [ ]:
# verdieping: elke beginmaand, de verandering tot de laatste maand, en de grafiek van die veranderingen
window_changes = None    # Series met de maand als index: verandering in punten van die maand tot de laatste maand
harshest_start = None    # de beginmaand met de grootste daling
kindest_start = None     # de beginmaand met de kleinste daling of de grootste stijging
n_rises = None           # hoeveel beginmaanden het aandeel laten lijken te stijgen
window_ax = None

In [ ]:
# CHECK 4
assert window_changes is not None and window_ax is not None, "window_changes of window_ax is nog leeg"
s = bus.set_index("month")["on_time_pct"]
expected = s.iloc[-1] - s.iloc[:-1]
assert len(window_changes) == len(expected), "één verandering voor elke maand behalve de laatste"
assert (abs(pd.Series(window_changes).values - expected.values) < 0.01).all(), "elke waarde is de laatste maand min de beginmaand, in punten"
assert pd.Timestamp(harshest_start) == expected.idxmin() and pd.Timestamp(kindest_start) == expected.idxmax()
assert n_rises == int((expected > 0).sum())
assert window_ax.get_ylim()[0] <= expected.min() and window_ax.get_ylim()[1] >= 0, "staven vanaf nul: de as moet nul en elke staaf bevatten"
print("CHECK 4 geslaagd: strengste begin", expected.idxmin().strftime("%Y-%m"), round(expected.min(), 1), "punten; mildste begin", expected.idxmax().strftime("%Y-%m"), round(expected.max(), 1), "punten")
print("Beginmaanden die een stijging laten zien:", int((expected > 0).sum()), "van", len(expected))
print("Verschil tussen de twee koppen:", round(expected.max() - expected.min(), 1), "punten")

**Kop vanaf het strengste begin (dubbelklik om te schrijven):**

**Kop vanaf het mildste begin:**

**Een zin die een zorgvuldige lezer kan vertrouwen:**